# 01 — Data overview

**Child Mind Institute — Problematic Internet Use**

Цель ноутбука — зафиксировать базовую структуру табличных данных до полноценного EDA:

- размеры `train` и `test`;
- типы данных;
- уникальность `id`;
- различия между наборами колонок;
- доступные model features;
- target-related / leakage-признаки;
- структуру `data_dictionary.csv`.

Actigraphy/parquet данные здесь намеренно не используются.


## 1. Imports and configuration

In [ ]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)

TARGET = "sii"
ID_COL = "id"


## 2. Paths and data loading

In [ ]:
# Работает при запуске Jupyter как из корня проекта, так и из project/notebooks/
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"

TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"
DATA_DICT_PATH = DATA_DIR / "data_dictionary.csv"

for path in [TRAIN_PATH, TEST_PATH, DATA_DICT_PATH]:
    if not path.exists():
        raise FileNotFoundError(f"File not found: {path.resolve()}")

print("Project root:", PROJECT_ROOT.resolve())
print("Data directory:", DATA_DIR.resolve())


In [ ]:
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
data_dict = pd.read_csv(DATA_DICT_PATH)

print("Train:", train.shape)
print("Test:", test.shape)
print("Data dictionary:", data_dict.shape)

display(train.head())


## 3. Dataset overview

Проверяем размеры выборок и корректность идентификаторов участников.


In [ ]:
overview = pd.DataFrame({
    "dataset": ["train", "test"],
    "rows": [len(train), len(test)],
    "columns": [train.shape[1], test.shape[1]],
    "unique_ids": [train[ID_COL].nunique(), test[ID_COL].nunique()],
    "duplicated_ids": [
        train[ID_COL].duplicated().sum(),
        test[ID_COL].duplicated().sum(),
    ],
})

display(overview)


### Data types

In [ ]:
dtype_summary = pd.DataFrame({
    "train": train.dtypes.value_counts(),
    "test": test.dtypes.value_counts(),
}).fillna(0).astype(int)

display(dtype_summary)


## 4. Train/test schema comparison

Признаки, отсутствующие в `test`, нельзя использовать как обычные model features.

Для дальнейшей работы безопаснее формировать набор входных признаков непосредственно из `test.columns`.


In [ ]:
train_only = sorted(set(train.columns) - set(test.columns))
test_only = sorted(set(test.columns) - set(train.columns))
common_cols = sorted(set(train.columns) & set(test.columns))

print(f"Common columns: {len(common_cols)}")
print(f"Train-only columns: {len(train_only)}")
print(f"Test-only columns: {len(test_only)}")

print("\nTrain-only:")
print(train_only)

print("\nTest-only:")
print(test_only)


## 5. Target and leakage-related columns

`PCIAT-PCIAT_Total` используется для получения `sii`, а PCIAT-поля отсутствуют в test.

Поэтому:

- `sii` — target;
- `id` — идентификатор, а не обычный model feature;
- все `PCIAT-*` нужно исключить из model features;
- основной набор model features удобно определять через `test.columns`.


In [ ]:
PCIAT_COLS = [c for c in train.columns if c.startswith("PCIAT")]
MODEL_FEATURES = [c for c in test.columns if c != ID_COL]

print("PCIAT columns:", len(PCIAT_COLS))
print("Model features available in test:", len(MODEL_FEATURES))

print("\nPCIAT columns:")
print(PCIAT_COLS)


In [ ]:
# Sanity checks
assert TARGET in train.columns, f"{TARGET!r} is absent from train"
assert TARGET not in test.columns, f"{TARGET!r} unexpectedly appears in test"
assert ID_COL in train.columns and ID_COL in test.columns

leakage_in_features = sorted(set(PCIAT_COLS) & set(MODEL_FEATURES))
print("PCIAT columns accidentally present in MODEL_FEATURES:", leakage_in_features)


## 6. Data dictionary

In [ ]:
display(data_dict.head(20))

print("Dictionary columns:")
print(data_dict.columns.tolist())


In [ ]:
instrument_summary = (
    data_dict["Instrument"]
    .value_counts(dropna=False)
    .rename_axis("Instrument")
    .to_frame("n_fields")
)

display(instrument_summary)


### Dictionary coverage

Проверяем, какие поля из train описаны в словаре и есть ли записи словаря, отсутствующие в данных.


In [ ]:
dictionary_fields = set(data_dict["Field"].dropna())
train_fields = set(train.columns)

not_in_dictionary = sorted(train_fields - dictionary_fields - {ID_COL, TARGET})
dictionary_only = sorted(dictionary_fields - train_fields)

print("Train fields without dictionary entry:", len(not_in_dictionary))
print(not_in_dictionary)

print("\nDictionary fields absent from train:", len(dictionary_only))
print(dictionary_only)
